# Étape D1 — Sélection des segments MGB-2 (Jeux D et A)

Ce notebook constitue la fine interface d'exécution pour l'étape D1.
Toute la logique de filtrage, de transcription ASR, de gestion des quotas et de reprise
est centralisée dans le module [`pipeline_mgb2.py`](file:///code/donnees/pipeline_mgb2.py).

### Modes d'exécution :
1. `MODE_REEL = False` (SIMULATION) : test hors ligne sans réseau ni GPU.
2. `MODE_REEL = True` (ESSAI RÉEL / COMPLET) : streaming officiel Kaldi MGB-2 et Whisper ASR réel.


In [ ]:
import os
import sys
from pathlib import Path

# Résolution des modules du projet
for p in [Path.cwd(), Path.cwd().parent, *Path.cwd().parents]:
    if (p / "code" / "donnees" / "pipeline_mgb2.py").exists():
        if str(p / "code" / "donnees") not in sys.path:
            sys.path.insert(0, str(p / "code" / "donnees"))
        break

from pipeline_mgb2 import executer_d1_selection
from seuils import charger_seuils

seuils = charger_seuils()
print("[INFO] Seuils officiels chargés depuis seuils.yaml.")

# Configuration de l'exécution
MODE_REEL = False               # False: simulation hors ligne | True: exécution réelle MGB-2 + Whisper
MAX_SEGMENTS = 20               # 20 pour essai réel, jusqu'à 50 000 en production Kaggle
MODELE_ASR = "openai/whisper-tiny"  # Whisper tiny pour essai local, large-v3 sur GPU Kaggle
REPARTITION = "alterne"         # 'alterne' pour essai (alimente D et A) | 'quota' pour grand volume

print(f"[D1 INTERFACE] Mode réel : {MODE_REEL} | Max segments : {MAX_SEGMENTS} | ASR : {MODELE_ASR}")


In [ ]:
# Exécution de D1 via pipeline_mgb2.py (source unique de vérité)
resultats = executer_d1_selection(
    mode_reel=MODE_REEL,
    max_segments=MAX_SEGMENTS,
    modele_ecoute=MODELE_ASR,
    repartition_mode=REPARTITION,
)

stats = resultats["stats"]
print("\n=== BILAN EXÉCUTION D1 ===")
print(f"  Entrée examinée      : {stats['entree']}")
print(f"  Déjà traités (reprise) : {stats['deja_traites']}")
print(f"  Retenus Jeu D        : {stats['gardes_d']}")
print(f"  Retenus Jeu A        : {stats['gardes_a']}")
print(f"  Rejetés              : {stats['rejets']}")
print(f"  Motifs de rejets     : {stats['rejets_par_motif']}")
print(f"  Durée audio totale   : {stats['duree_audio_totale_s']}s")
print(f"  Temps inférence ASR  : {stats['temps_inference_total_s']:.2f}s")
print(f"  Facteur Temps Réel   : {stats['facteur_temps_reel_rtf']}")
print(f"\nSorties :")
print(f"  - D sélection : {resultats['chemin_d']}")
print(f"  - A alignement: {resultats['chemin_a']}")
print(f"  - Rejets D1   : {resultats['chemin_rejets']}")
print(f"  - Métriques   : {resultats['chemin_stats']}")
